# 🕵️ Rewriting Legal Documents

Rewriting legal text (TAB dataset) with a domain-specific privacy goal
and custom entity labels tailored for legal proceedings.

#### 📚 What you'll learn

- Define domain-specific entity labels for legal text (case numbers, court names, etc.)
- Configure rewrite mode with legal-specific privacy goals
- Preview and run on court decision documents
- Triage flagged records with `needs_human_review`

> **Tip:** First time running notebooks? Start with
> [setup instructions](https://nvidia-nemo.github.io/Anonymizer/latest/tutorials/).

## ⚙️ Setup

- Check if your `NVIDIA_API_KEY` from [build.nvidia.com](https://build.nvidia.com) is registered for model access.
- Import `Detect` (for custom entity labels), `Rewrite`, and its config classes.
- `Anonymizer()` initializes with the default model provider -- no extra config needed.
- `configure_logging(enabled=False)` suppresses pipeline logs for cleaner output.
  Switch to `configure_logging(LoggingConfig.debug())` when troubleshooting.

In [2]:
import getpass
import os

if not os.getenv("NVIDIA_API_KEY"):
    key = getpass.getpass("Enter NVIDIA_API_KEY from build.nvidia.com: ").strip()
    if not key:
        raise RuntimeError("NVIDIA_API_KEY is required to run these notebooks.")
    os.environ["NVIDIA_API_KEY"] = key

In [3]:
from anonymizer import Anonymizer, AnonymizerConfig, AnonymizerInput, Detect, Rewrite, configure_logging, LoggingConfig
from anonymizer.config.rewrite import PrivacyGoal

configure_logging(LoggingConfig.default())

In [4]:
anonymizer = Anonymizer()

[16:41:39] [INFO] 🔧 Anonymizer initialized with 3 model configs
[16:41:39] [INFO]   |-- 🔎 detector:  gliner-pii-detector
[16:41:39] [INFO]   |-- ✅ validator: gpt-oss-120b
[16:41:39] [INFO]   |-- 🧩 augmenter: gpt-oss-120b


## 📦 Input data

- [TAB (Text Anonymization Benchmark)](https://github.com/NorskRegnesentral/text-anonymization-benchmark)
  legal documents -- court decisions containing names, dates, case numbers, and other legal identifiers.
- `LEGAL_ENTITY_LABELS` defines the domain-specific entity types to detect.
  This replaces the default label set with one tailored to legal text.

In [5]:
LEGAL_ENTITY_LABELS = [
    "first_name",
    "last_name",
    "court_name",
    "organization_name",
    "company_name",
    "prison_detention_facility",
    "street_address",
    "city",
    "state",
    "country",
    "date",
    "date_time",
    "time",
    "date_of_birth",
    "age",
    "email",
    "phone_number",
    "ssn",
    "unique_id",
    "legal_role",
    "case_number",
    "application_number",
    "monetary_amount",
    "sentence_duration",
    "nationality",
]

input_data = AnonymizerInput(
    source="../data/TAB_legal_sample25.csv",
    text_column="text",
    data_summary="Legal court decisions containing personal identifiers, case numbers, and institutional references",
)

## 🎛️ Configure

- `Detect(entity_labels=...)` overrides the default entity set with legal-specific labels.
- `PrivacyGoal` tells the rewriter what to **protect** (identifiers, case numbers,
  institutional references) and what to **preserve** (legal reasoning, statutory references,
  ruling structure).

In [6]:
config = AnonymizerConfig(
    detect=Detect(
        entity_labels=LEGAL_ENTITY_LABELS,
    ),
    rewrite=Rewrite(
        privacy_goal=PrivacyGoal(
            protect="All personal identifiers, case numbers, court names, and institutional references that could identify parties",
            preserve="Legal reasoning, procedural facts, statutory references, and the structure of the ruling",
        ),
        risk_tolerance="minimal",
        max_repair_iterations=3,
    ),
)

## 👁️ Preview

- Preview on a few records to check that legal entities are detected
  and the rewrite preserves the ruling's structure.

In [ ]:
preview = anonymizer.preview(
    config=config,
    data=input_data,
    num_records=3,
)

preview.display_record(0)

[16:41:39] [INFO] 📂 Loaded 25 records from ../data/TAB_legal_sample25.csv (column: 'text')
[16:41:39] [INFO] detection labels in scope: ['age', 'application_number', 'case_number', 'city', 'company_name', 'country', 'court_name', 'date', 'date_of_birth', 'date_time', 'email', 'first_name', 'last_name', 'legal_role', 'monetary_amount', 'nationality', 'organization_name', 'phone_number', 'prison_detention_facility', 'sentence_duration', 'ssn', 'state', 'street_address', 'time', 'unique_id']
[16:41:39] [INFO]   |-- 👀 Preview mode: processing 3 of 25 records
[16:41:39] [INFO] 🔍 Running entity detection on 3 records
[16:43:07] [INFO]   |-- 📋 Detection complete — 141 entities found across 3 records (0 failed) [87.2s]
[16:43:07] [INFO]   |-- labels: date=51, court_name=35, legal_role=10, nationality=7, last_name=7, organization_name=6, country=5, first_name=5, city=5, application_number=3, date_of_birth=3, monetary_amount=2, case_number=1, sentence_duration=1
[16:43:07] [INFO] ✏️ Running rewr

In [ ]:
preview.display_record(1)

Entity,Label,Sensitivity,Protection
29360/06,application_number,high,replace
Republic of Poland,country,low,leave_as_is
Polish,nationality,low,leave_as_is
Teresa,first_name,high,replace
Jerzak,last_name,high,replace
3 July 2006,date,medium,generalize
Agent,legal_role,low,leave_as_is
Wołąsiewicz,last_name,high,replace
Ministry of Foreign Affairs,organization_name,medium,replace
25 September 2007,date,medium,generalize


## 🚀 Full run

- `result.dataframe` has user-facing columns: rewritten text, scores, and the review flag.

In [ ]:
result = anonymizer.run(config=config, data=input_data)

result.dataframe.head()

[14:33:42] [INFO] 📂 Loaded 25 records from ../data/TAB_legal_sample25.csv (column: 'text')


[14:33:42] [INFO] detection labels in scope: ['age', 'application_number', 'case_number', 'city', 'company_name', 'country', 'court_name', 'date', 'date_of_birth', 'date_time', 'email', 'first_name', 'last_name', 'legal_role', 'monetary_amount', 'nationality', 'organization_name', 'phone_number', 'prison_detention_facility', 'sentence_duration', 'ssn', 'state', 'street_address', 'time', 'unique_id']


[14:33:42] [INFO] 🔍 Running entity detection on 25 records


[14:33:42] [INFO] 🎨 Creating Data Designer dataset


[14:33:42] [INFO] ✅ Validation passed


...

[15:24:06] [INFO]   |-- 🔧 column: 'needs_human_review'


/Users/lramaswamy/Documents/github/Anonymizer/src/anonymizer/engine/row_partitioning.py:42: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  pd.concat(list(parts), ignore_index=True)
[15:24:06] [INFO]   |-- 📋 Rewrite complete (2 failed) [2512.4s]


[15:24:06] [WARNING] 2 record(s) failed during pipeline processing.


[15:24:06] [INFO] 🎉 Pipeline complete — 25 records processed, 2 total failures


,text,text_rewritten,utility_score,leakage_mass,weighted_leakage_rate,any_high_leaked,needs_human_review
0,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,0.971429,0.0,0.0,False,False
1,PROCEDURE The case originated in an applicati...,Procedure The case originated in an applicati...,0.822222,0.0,0.0,False,False
2,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,0.90625,1.7,0.05414,True,True
3,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,0.928571,0.0,0.0,False,False
4,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,0.857143,0.0,0.0,False,False


In [ ]:
result.dataframe[["text_rewritten", "utility_score", "leakage_mass", "needs_human_review"]].head()

,text_rewritten,utility_score,leakage_mass,needs_human_review
0,PROCEDURE The case originated in an applicati...,0.971429,0.0,False
1,Procedure The case originated in an applicati...,0.822222,0.0,False
2,PROCEDURE The case originated in an applicati...,0.90625,1.7,True
3,PROCEDURE The case originated in an applicati...,0.928571,0.0,False
4,PROCEDURE The case originated in an applicati...,0.857143,0.0,False


## 🚩 Filter by review flag

- Records where automated metrics exceed thresholds are flagged for manual review.
- Use this to prioritize human attention on the records that need it most.
- See [Working with flagged records](../concepts/rewrite.md#working-with-flagged-records)
  for guidance on diagnosing and resolving flagged records.

In [ ]:
df = result.dataframe
flagged = df[df["needs_human_review"] == True]  # noqa: E712
print(f"{len(flagged)} of {len(df)} records flagged for human review")
flagged.head()

6 of 23 records flagged for human review


,text,text_rewritten,utility_score,leakage_mass,weighted_leakage_rate,any_high_leaked,needs_human_review
2,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,0.90625,1.7,0.05414,True,True
8,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,0.96,3.052,0.127167,True,True
11,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,1.0,0.9,0.046392,True,True
12,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,0.866667,2.97,0.086589,True,True
13,PROCEDURE The case originated in an applicati...,PROCEDURE The case originated in an applicati...,0.981818,3.1,0.133621,True,True


## ⏭️ Next steps

- **[🔍 Inspecting Detected Entities](02_inspecting_detected_entities.ipynb)** --
  debug what the detection pipeline found before rewriting.
- **Try it on your own data!** Swap in your CSV, define entity labels for your
  domain, and set a `PrivacyGoal` that fits -- you've got all the building blocks.